# 05 — Business Insights

Pulls together the query results, the visualizations, and the churn model into the headline findings. Every number below is read directly from `reports/query_results.json` and `reports/churn_model_metrics.json` — nothing here is hand-typed.

In [1]:
import pandas as pd
import json

results = json.load(open('reports/query_results.json'))
churn = json.load(open('reports/churn_model_metrics.json'))

## Finding 1 — Marketing spend is efficient in aggregate, but wildly uneven by channel

In [2]:
cac = pd.DataFrame(results['Q13']['sample'])
eff = pd.DataFrame(results['Q14']['sample'])
print('CAC by channel ($):')
print(cac[['channel','cac']].to_string(index=False))
print()
print('Revenue generated per $1 spent, by channel:')
print(eff[['channel','revenue_per_dollar_spent']].to_string(index=False))

CAC by channel ($):
    channel   cac
     Direct  2.53
    Organic  4.12
      Email  9.91
     Social 32.86
  Affiliate 36.57
Paid Search 46.09

Revenue generated per $1 spent, by channel:
    channel  revenue_per_dollar_spent
     Direct                    717.02
    Organic                    459.89
      Email                    197.91
     Social                     58.44
  Affiliate                     50.53
Paid Search                     43.38


Direct and Organic are almost free and return $700 / $460 of revenue per dollar of attributable spend — but they only bring in a few thousand customers a year, so they can't be scaled past their natural ceiling. Paid Search is 18x more expensive per customer than Direct, but it's still the single largest revenue driver because of volume. **The actionable read: shift incremental budget toward Email and Organic first (cheap, under-invested), but don't defund Paid Search — cut it and you lose the volume channel, not just an inefficient one.**

## Finding 2 — Enterprise customers trade margin for scale

In [3]:
seg = pd.DataFrame(results['Q18']['sample'])
print(seg.to_string(index=False))

       segment  total_revenue  gross_margin_pct  avg_discount_pct  revenue_per_customer
Small Business     8524031.97             27.24             10.12               3254.69
    Enterprise     7309082.51             23.87             14.01              10265.57
      Consumer     7305841.39             30.49              5.70                926.31


Enterprise customers get the deepest average discount (14.0%) and the thinnest gross margin (23.9%) of the three segments — but each one is worth **11x** more revenue than a Consumer customer ($10,266 vs $926). Small Business sits in between and is actually the single largest revenue segment overall. This is a normal B2B/B2C tradeoff, not a problem — but it means blended "average margin" headline numbers hide a real decision: margin-protection policy should differ by segment, not be a single company-wide discount cap.

## Finding 3 — Fashion has the highest return rate, Electronics the highest return cost

In [4]:
ret = pd.DataFrame(results['Q15']['sample'])
print(ret.to_string(index=False))

   category  total_orders  total_returns  return_rate_pct  total_refunded
    Fashion         11976           1778            14.85       371063.51
Electronics          8758            899            10.26      1082287.76
    Fitness          5551            436             7.85       140034.02
       Home          9555            608             6.36       239883.20
    Outdoor          4806            281             5.85        97123.74
     Beauty          7092            347             4.89        33872.77
     Office          4768            167             3.50        43469.97
    Grocery          6432             73             1.13         3488.14


Fashion returns 14.9% of orders (sizing/fit, expected for apparel) but Electronics — despite a lower 10.3% return rate — refunds nearly 3x more in dollar terms because unit prices are so much higher. **A return-rate KPI alone would send attention to the wrong category; the dollar-impact view sends it to Electronics.**

## Finding 4 — Churn model: a real but modest signal, reported honestly

In [5]:
print(f"ROC-AUC: {churn['roc_auc']}")
print(f"Churn rate in the 180-day observation window: {churn['churn_rate_in_window']*100:.1f}%")
print(f"Eligible customers scored: {churn['feature_rows']:,}")
print()
print('Top features:')
for k, v in list(churn['feature_importance'].items())[:5]:
    print(f'  {k}: {v}')

ROC-AUC: 0.5822
Churn rate in the 180-day observation window: 19.0%
Eligible customers scored: 7,348

Top features:
  segment_enc: 0.1485
  avg_discount: 0.1428
  avg_order_value: 0.1364
  monetary: 0.1224
  total_quantity: 0.1121


An ROC-AUC around 0.58 is a real, above-random signal — not a strong one. Worth stating plainly rather than dressing it up: recency and monetary value carry most of the predictive weight, which is the classic RFM pattern, but the model doesn't separate "about to churn" customers from "always ordered infrequently" customers very cleanly. **Honest conclusion: this is good enough to rank customers for a retention-outreach queue (better than no prioritization), not good enough to justify automated action (like revoking a loyalty tier) on its prediction alone.** In production, the next step would be adding session/browsing behavior — not available in transactional data alone — before trusting this for anything higher-stakes.

## Finding 5 — RFM segmentation shows where retention dollars matter most

In [6]:
seg_summary = results['Q10']['sample']
for row in seg_summary:
    print(f"{row['rfm_segment']:<28} {row['customers']:>6} customers   avg lifetime revenue ${row['avg_lifetime_revenue']:,.2f}")

Champions                      2729 customers   avg lifetime revenue $4,697.60
At Risk (was frequent)         2331 customers   avg lifetime revenue $2,642.52
Loyal                          1865 customers   avg lifetime revenue $888.76
Lost                           3279 customers   avg lifetime revenue $626.19
Developing                     1014 customers   avg lifetime revenue $442.48


**At Risk (was frequent)** customers are worth 4x more on average than **Lost** customers and still represent real historical spend (~$2,643 avg) — this is the highest-leverage group for a win-back campaign, since they've already proven they'll buy repeatedly, unlike **Developing** customers who haven't established a pattern yet.

## Summary of headline metrics

In [7]:
q1 = pd.DataFrame(results['Q1']['sample'])
print(f"Total revenue (last 3 months of data, sample rows shown): see full series in Q1")
print()
print('This project intentionally reports weak/negative findings alongside strong ones ' 
      '(the modest churn AUC, the Enterprise margin tradeoff) because a portfolio piece ' 
      'that only ever finds good news isn\'t credible.')

Total revenue (last 3 months of data, sample rows shown): see full series in Q1

This project intentionally reports weak/negative findings alongside strong ones (the modest churn AUC, the Enterprise margin tradeoff) because a portfolio piece that only ever finds good news isn't credible.
